# Medical Cost Dataset — Analytics & Linear Regression

This notebook loads and cleans the medical cost dataset, performs exploratory analysis, trains a Linear Regression model using **8 selected inputs**, evaluates it with regression metrics, and saves `medical_cost_model.sav`.

Target: `annual_medical_cost`. Put `medical_cost_prediction_dataset.csv` beside this notebook before running.

In [ ]:
# Optional setup: %pip install pandas numpy matplotlib seaborn scikit-learn joblib openpyxl
import os, warnings, joblib
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
warnings.filterwarnings("ignore")
DATA_PATH="medical_cost_prediction_dataset.csv"
OUTPUT_DIR="outputs"; os.makedirs(OUTPUT_DIR,exist_ok=True)
TARGET="annual_medical_cost"
FEATURES=["previous_year_cost","age","bmi","smoker","hospital_admissions","doctor_visits_per_year","medication_count","insurance_coverage_pct"]
RANDOM_STATE=42


## 1. Load and inspect data

In [ ]:
df=pd.read_csv(DATA_PATH)
print("Dataset shape:",df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)


## 2. Missing values, duplicates, and cleaning

In [ ]:
missing_before=df.isna().sum()
duplicates=int(df.duplicated().sum())
clean=df.drop_duplicates().copy()
for col in clean.columns:
    if clean[col].isna().any():
        if pd.api.types.is_numeric_dtype(clean[col]):
            clean[col]=clean[col].fillna(clean[col].median())
        else:
            mode=clean[col].mode(dropna=True)
            clean[col]=clean[col].fillna(mode.iloc[0] if len(mode) else "Unknown")
print("Duplicate rows removed:",duplicates)
print("Missing cells before:",int(missing_before.sum()))
print("Missing cells after:",int(clean.isna().sum().sum()))
display(pd.DataFrame({"Missing Before":missing_before,"Missing After":clean.isna().sum()}))
clean.to_csv(os.path.join(OUTPUT_DIR,"cleaned_medical_cost_dataset.csv"),index=False)


## 3. Exploratory Data Analysis

In [ ]:
display(clean.describe(include="all").T)
plt.figure(figsize=(7,4))
sns.histplot(clean[TARGET],kde=True)
plt.title("Annual Medical Cost Distribution"); plt.tight_layout(); plt.show()

numeric=clean.select_dtypes(include=np.number)
plt.figure(figsize=(12,8))
sns.heatmap(numeric.corr(),cmap="coolwarm",center=0)
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

for col in ["smoker","gender","insurance_type","city_type","physical_activity_level"]:
    if col in clean.columns:
        avg=clean.groupby(col,dropna=False)[TARGET].mean().sort_values(ascending=False)
        display(avg.rename("Mean Annual Medical Cost").to_frame())
        avg.plot(kind="bar",figsize=(7,4),title=f"Mean Annual Medical Cost by {col}")
        plt.ylabel("Mean annual cost"); plt.tight_layout(); plt.show()


## 4. Select prediction target and 8 inputs

**Target:** `annual_medical_cost` (continuous), so this is a regression task.

**Inputs:** previous_year_cost, age, bmi, smoker, hospital_admissions, doctor_visits_per_year, medication_count, insurance_coverage_pct. These cover prior cost, demographic/health status, utilization, and coverage. A short input form is easier to use; feature relevance should be validated on the dataset and in the intended application.

In [ ]:
missing=[c for c in FEATURES+[TARGET] if c not in clean.columns]
if missing: raise ValueError(f"Required columns not found: {missing}")
X=clean[FEATURES].copy()
y=pd.to_numeric(clean[TARGET],errors="raise")
display(pd.DataFrame({"Feature":FEATURES,"Type":[str(X[c].dtype) for c in FEATURES]}))


## 5. Split 80/20 and train Linear Regression

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE)
numeric_features=X.select_dtypes(include=np.number).columns.tolist()
categorical_features=[c for c in X.columns if c not in numeric_features]
preprocessor=ColumnTransformer([
 ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numeric_features),
 ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical_features)
])
model=Pipeline([("preprocessor",preprocessor),("regressor",LinearRegression())])
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
print("Training rows:",len(X_train),"Testing rows:",len(X_test))


## 6. Evaluate model

In [ ]:
metrics={
 "R²":r2_score(y_test,y_pred),
 "MAE":mean_absolute_error(y_test,y_pred),
 "RMSE":np.sqrt(mean_squared_error(y_test,y_pred)),
 "MAPE (%)":mean_absolute_percentage_error(y_test,y_pred)*100
}
metrics_df=pd.DataFrame(metrics.items(),columns=["Metric","Value"])
display(metrics_df)
plt.figure(figsize=(6,6))
plt.scatter(y_test,y_pred,alpha=.5)
low=min(y_test.min(),y_pred.min()); high=max(y_test.max(),y_pred.max())
plt.plot([low,high],[low,high],linestyle="--")
plt.xlabel("Actual Annual Medical Cost"); plt.ylabel("Predicted Annual Medical Cost")
plt.title("Actual vs Predicted"); plt.tight_layout(); plt.show()

residuals=y_test-y_pred
plt.figure(figsize=(7,4))
sns.histplot(residuals,kde=True)
plt.title("Prediction Residuals"); plt.xlabel("Actual - Predicted"); plt.tight_layout(); plt.show()


## 7. Save model and reports

In [ ]:
joblib.dump(model,os.path.join(OUTPUT_DIR,"medical_cost_model.sav"))
metrics_df.to_csv(os.path.join(OUTPUT_DIR,"model_metrics.csv"),index=False)
pd.DataFrame({"Feature":FEATURES}).to_csv(os.path.join(OUTPUT_DIR,"model_features.csv"),index=False)
missing_report=pd.DataFrame({"Column":df.columns,
 "Missing Before":[int(df[c].isna().sum()) for c in df.columns],
 "Missing After":[int(clean[c].isna().sum()) for c in df.columns]})
with pd.ExcelWriter(os.path.join(OUTPUT_DIR,"analytics_summary.xlsx"),engine="openpyxl") as writer:
    metrics_df.to_excel(writer,sheet_name="Model Metrics",index=False)
    missing_report.to_excel(writer,sheet_name="Missing Values",index=False)
    clean.describe(include="all").T.to_excel(writer,sheet_name="Data Summary")
print("Saved model:",os.path.abspath(os.path.join(OUTPUT_DIR,"medical_cost_model.sav")))
print("Saved outputs in:",os.path.abspath(OUTPUT_DIR))
